# Démo : Agent dans Gradio

**Contexte :** assistant du service client de Lagune Banque.

Cette démonstration relie l’agent du TP1 à une interface de chat. Les appels d’outils, leurs arguments, leurs résultats et leur durée sont affichés dans des blocs dédiés.

**Sommaire**
- [Configuration](#configuration)
- [Génération de la réponse](#generation-de-la-reponse)
- [L'interface](#linterface)

## <a id="configuration"></a>Configuration

In [ ]:
import sys
from pathlib import Path

RACINE = next(p for p in [Path("."), Path(".."), Path("../.."), Path("../../.."), Path("starter"), Path("../../starter")] if (p / "code_partage").exists()).resolve()
sys.path[:0] = [str(RACINE / "code_partage"), str(RACINE / "03_agents" / "socle")]

import inspect

import chat
import front
from base import KOUASSI, Context, new_model, reset_db
from config import prepare
from langchain.agents import create_agent
from tools import SYSTEM_PROMPT, TOOLS

In [ ]:
prepare()
reset_db()
agent = create_agent(new_model(), TOOLS, system_prompt=SYSTEM_PROMPT, context_schema=Context)
print("Tools de l'agent :", [t.name for t in TOOLS])

## <a id="generation-de-la-reponse"></a>Génération de la réponse

`agent.stream(..., stream_mode="updates")` renvoie un événement à chaque étape du graphe de l'agent : le modèle décide d'appeler un tool, puis le tool répond. `chat.respond` transforme chaque événement en bloc :

- un appel de tool devient un bloc au statut `pending` ;
- son résultat le remplace par le même bloc au statut `done`, avec la durée ;
- la réponse finale arrive en dernier ;
- si l'agent est suspendu avant une action sensible, le chat l'indique.

**Syntaxe.** `agent.stream(entrée, config, context=..., stream_mode="updates")` est un générateur, là où `invoke` ne renvoie que l'état final. Chaque événement est un dictionnaire `{nœud: mise à jour}`. L'agent a deux nœuds : `model`, dont le message est un `AIMessage` (avec `tool_calls` quand il appelle un tool), et `tools`, dont les messages sont des `ToolMessage`.

In [ ]:
print(inspect.getsource(chat.respond))

Test de la fonction, sans l'interface. Gradio fournit le vrai `request` pendant l'utilisation ; ici, un objet minimal le remplace.

In [ ]:
class Requete:
    session_hash = "test0000"


for sortie in chat.respond(agent)(KOUASSI, [], "CLI-0002", Requete()):
    pass
print([m.metadata["title"] for m in sortie if m.metadata])
print(sortie[-1].content)

## <a id="linterface"></a>L'interface

La liste déroulante est un champ supplémentaire : Gradio en passe la valeur à la fonction de réponse, après le message et l'historique. Le client identifié est choisi par l'application, pas par le modèle.

In [ ]:
demo = chat.create(agent)
front.launch(demo)

Essayez : `J'ai remboursé mon prêt hier via Lagune Money, mais mon dossier indique toujours un impayé.`

L'agent enchaîne en général les prêts du client, l'échéancier, la transaction Lagune Money, l'incident de la plateforme, la procédure en vigueur, puis la proposition de régularisation, qui reste en attente d'un conseiller.

Les mêmes appels, avec tokens, latence et coût de chaque étape, se lisent dans Langfuse : démo 3 du module.

In [ ]:
demo.close()

## Contrôle

In [ ]:
titres = [m.metadata["title"] for m in sortie if m.metadata]
assert any("check_lm_transaction" in t for t in titres), "l'agent n'a pas vérifié la transaction"
assert sortie[-1].content, "pas de réponse finale"
print("OK")